In [12]:
import pandas as pd

# ==========================================
# STORAGE
# ==========================================
all_chunks = []
print("Processing crop production dataset...")

# ==========================================
# READ DATASET IN CHUNKS
# ==========================================
for chunk in pd.read_csv(
    "../datasets/yield/crop_production.csv",
    engine='python',
    chunksize=50000
):
    # ======================================
    # CLEAN COLUMN NAMES
    # ======================================
    chunk.columns = chunk.columns.str.strip()

    # ======================================
    # KEEP REQUIRED COLUMNS
    # ======================================
    chunk = chunk[[
        'State_Name',
        'District_Name',
        'Crop_Year',
        'Crop',
        'Area',
        'Production'
    ]]

    # ======================================
    # REMOVE NULL VALUES
    # ======================================
    chunk = chunk.dropna()

    # ======================================
    # CLEAN TEXT
    # ======================================
    chunk['State_Name'] = chunk['State_Name'].astype(str)
    chunk['District_Name'] = chunk['District_Name'].astype(str)
    chunk['Crop'] = chunk['Crop'].astype(str)

    # ======================================
    # SAFE FILTERING
    # ======================================
    filtered = chunk[
        chunk['State_Name']
        .str.contains('Uttar Pradesh', case=False, na=False)
        &
        chunk['Crop']
        .str.contains('Wheat', case=False, na=False)
        &
        (chunk['Crop_Year'] >= 2005)
        &
        (chunk['Crop_Year'] <= 2014)
    ]

    # ======================================
    # DEBUG
    # ======================================
    if len(filtered) > 0:
        print("FOUND:", len(filtered))
        all_chunks.append(filtered)

# ==========================================
# COMBINE ALL CHUNKS
# ==========================================
df = pd.concat(all_chunks, ignore_index=True)
print("\nFINAL DATA:")
print(df.head())
print("\nTOTAL ROWS:")
print(len(df))

# ==========================================
# CALCULATE YIELD
# ==========================================
df['Yield'] = df['Production'] / df['Area']

# ==========================================
# KEEP REQUIRED COLUMNS
# ==========================================
yield_df = df[[
    'District_Name',
    'Crop_Year',
    'Yield'
]].copy()

# ==========================================
# STANDARDIZE DISTRICT NAMES & SHIFT YEAR
# ==========================================
yield_df['District_Name'] = yield_df['District_Name'].str.strip().str.title()
yield_df['District_Name'] = yield_df['District_Name'].replace({
    'Allahabad': 'Prayagraj',
    'Kanpur Nagar': 'Kanpur'
})

yield_df.rename(
    columns={
        'Crop_Year': 'Year'
    },
    inplace=True
)
yield_df['Year'] = yield_df['Year'] + 10

# ==========================================
# SAVE CSV
# ==========================================
yield_df.to_csv(
    "../datasets/yield/full_up_yield.csv",
    index=False
)
print("\nFULL UP YIELD DATASET SAVED!")
print(yield_df.head())
